In [ ]:
!pip install pandas openai python-dotenv tqdm openpyxl
!pip show openai

In [ ]:
import os
import pandas as pd
import openai
from dotenv import load_dotenv
from tqdm import tqdm
from openai import OpenAI

In [ ]:
# .env file loading (api key)
load_dotenv()

# api key defining
openai.api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=openai.api_key)

In [ ]:
# prompt
prompt_template = """
You are a classification assistant.
Your task is to categorize academic papers based only on their title and abstract.

You will be provided with:
•⁠  ⁠Title: the paper's title
•⁠  ⁠Abstract: the paper's abstract

Classify the paper into exactly one of the following three types:

type_1: The paper focuses on nurses conducting research.

type_2: The paper focuses on nurses implementing research findings in clinical settings (i.e., Evidence-Based Practice - EBP).

type_3: The paper does not focus exclusively on either conducting or implementing research, but instead discusses both aspects more or less equally.

If uncertain, choose the type that best reflects the core emphasis of the abstract and title.

Now, analyze and classify the following based on the title and abstract provided:
_
Title: {title}
Abstract: {abstract}
_

Return your answer in exactly the following format:

Decision: <type_1 / type_2 / type_3>
Explain: <Brief explanation referencing the relevant type definition>
"""


In [ ]:
def classify_paper(title: str, abstract: str):
    # prompt defining with 'title' and 'abstract'
    prompt = prompt_template.format(title=title, abstract=abstract)

    try:
        response = client.chat.completions.create(
            model="gpt-4o",  # USING GPT-4o !
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,  # for determination
            max_tokens=150    # limit the answer's length

        )

        answer = response.choices[0].message.content.strip()

        decision = ""
        explanation = ""
        if "**Decision:**" in answer:

            parts = answer.split("**Explain:**")
            decision_part = parts[0]
            explanation_part = parts[1] if len(parts) > 1 else ""

            decision = decision_part.replace("**Decision:**", "").strip()
            explanation = explanation_part.strip()
        elif "Decision:" in answer:

            parts = answer.split("Explanation:") if "Explanation:" in answer else answer.split("Explain:")
            decision_part = parts[0]
            explanation_part = parts[1] if len(parts) > 1 else ""
            decision = decision_part.replace("Decision:", "").strip()
            explanation = explanation_part.strip()
        else:

            explanation = answer

        return decision, explanation
    except Exception as e:
        print(f"Error classifying paper with title: {title}\nError: {e}")
        return None, None



In [ ]:
# paper's excel loading
df = pd.read_excel("485_not_systematic_final_4.xlsx")
print("Columns in the csv file:", df.columns)


Columns in the csv file: Index(['PubMed ID', 'DOI', 'Title', 'Year', 'Abstract', 'Author Keywords',
       'english', 'Decision', 'Decision Type', 'Type_article',
       ...
       'Unnamed: 1180', 'Unnamed: 1181', 'Unnamed: 1182', 'Unnamed: 1183',
       'Unnamed: 1184', 'Unnamed: 1185', 'Unnamed: 1186', 'Unnamed: 1187',
       'Unnamed: 1188', 'Unnamed: 1189'],
      dtype='object', length=1190)


In [ ]:
decisions = []
explanations = []
for idx, row in df.iterrows():
    title = row["Title"]
    abstract = row["Abstract"]
    dec, exp = classify_paper(title, abstract)
    decisions.append(dec)
    explanations.append(exp)

In [ ]:
# adding new columns to the excel file
df["Classification"] = decisions
df["Explanation"] = explanations

# first 5 lines for check
df.head()
# saving the new
output_file = "485_classified_with_type_5.xlsx"
df.to_xlsx(output_file, index=False)
print(f"Results saved to {output_file}")


Results saved to 487_classified.csv
